# SE2_data_process
This notebook will be used to generate bathymetric surfaces from previously downloaded hydrographic survey point cloud data. Four interpolation methods are implemented:
1. Triangulated Irregular Network (TIN)
2. Natural Neighbor
3. Ordinary kriging (OK)
4. Regrssion Kriging (RK)

Additional preprocessing steps includes a mean data decimation to reduce the point clouds to 15,000 points to emulate a common approach in bathymetric point cloud processing, and to maintain Kriging numerical stability.

# Overview:
## Input data description:
Data is stored in .gdb file format, which is read using *fiona* and *geopandas*. Because eHydro covers many regions of the US, data is stored in varying forms of the State Plane Coordinate system for the region the data was aquired. Within the .gdb, three layers may be used: *SurveyPoint* for a pre-processed point clouds, *SurveyPoint_HD* for a pre-processed dense point clouds, and *SurveyJob* for a boundary shapefile for the survey. Within the two available point cloud layers, the xLocation, yLocation, and Depth (Z_use) are reported relative to a local Mean Lowest Low Water (MLLW) datum.

## Outputs:
Each point cloud will be processed into Cloud Optimized GeoTIFF (COG) format. The COGs will be saved in their original CRS relative to the corresponding .gdb file which contained the point cloud. The spatial resolution for each COG is 10 feet, to correspond with the inputs used in the US Army Corps of Engineers Corps Shoaling Analysis Tool (CSAT). Additionally, with each COG, a validation CSV is provided which stores the sampled depth from the point cloud, the predicted depth, and residual at each point within the raw point cloud. This CSV will be used to analyze bias and compare interpolation performance.

## Modelling framework:
Processed data will be compatible with most software, as the two data outputs are stored in open COG and CSV formats.

## Data processing steps:
1. Data decimation to a target number of points using *Verde.BlockReduce*. A reduction method may be specified, but Median is default
2. Specify and fit multiple variogram shape functions, using Cressie or Matheron weighted least squares approach to solve for nugget, range, and sill. The Aikaike Information Criterion is used to select the best fit empirical variogram for the kriging model
3. TIN, Natural Neighbor, Ordinary Kriging, and Regression Kriging employed on decimated point cloud to generate COGs
4. Raw dense point clouds sampled along predicted rasters for validation

## How to use this notebook:
This notebook will read data from ```raw_data``` folder, or a specified data storage folder. A user may specify the output grid resolution, the specific variogram models to test, which least square solution to use for empirical variogram fitting, what reduction method to use for data decimation, and whether or not to plot the COGs as they are generated.



In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.insert(0, os.path.abspath('../../src'))
import pandas as pd
import geopandas as gpd
from pathlib import Path
import gstools as gs
import fiona

from interpolators.TIN import TIN
from interpolators.NaturalNeighbor import NaturalNeighbor
from interpolators.KNeighbors_IDW import IDW
from interpolators.RBF import RBF
from interpolators.isoAutoKrige import isoAutoKrige

import processing_help

import warnings
# remove future warning of spline mindist since we aren't even providing it anyways
warnings.filterwarnings('ignore', message='.*mindist parameter.*', category=FutureWarning)     

In [ ]:
# DATA_DIR = Path(os.getcwd()).parent.parent / 'raw_data'

# for local data:
DATA_DIR = Path('/mnt/teamgroup/AutoDBM/source_data')

diverse_surveys= pd.read_csv(Path(os.getcwd()).parent.parent / 'analysis_data' / 'survey_characteristics.csv')
diverse_surveys

# survey_gdbs = list(DATA_DIR.glob('*/*/*.gdb'))
# len(survey_gdbs)

In [ ]:
survey_gdbs = []

for survey in diverse_surveys.survey.values:
    survey_gdbs.append(list(DATA_DIR.glob(f'*/*/*{survey}.gdb'))[0])

print(len(survey_gdbs))

In [ ]:
vario_models = {
    "Gaussian": gs.Gaussian,
    "Exponential": gs.Exponential,
    "Matern": gs.Matern,
    "Stable": gs.Stable,
    "Rational": gs.Rational,
    "Circular": gs.Circular,
    "Spherical": gs.Spherical,
    "SuperSpherical": gs.SuperSpherical,
    "JBessel": gs.JBessel,
}

# least square estimation scheme for automatic variogram fitting
# 'cressie' or 'matheron'. Cressie tends to be more robust to outliers
estimator = 'cressie'

# all data from eHydro is projected in state plance coordinates
# measurements units referenced to US Survey Feet
grid_res = 10.0 # ft

# for example on a not so big computer
# NOTE: Kriging is very computationally heavy, as is global spline fitting. A PC with high availabel RAM is preferred
# even with ~ 90G of RAM, I can only make it to 15k points sometimes
num_pts= 10000

reduction_method = 'median'     # median to remove outliers, min to get shoalest data for more accurately representing potential shoals. Will test and compare results using median, mean, minimum and nav_safe to know which is best

plots = False               # if the outputs of the interpolation methods will be showcased in this notebook using Matplotlib

# Generate surfaces for all 100 surveys

In [ ]:
bad_surveys = []

for survey in survey_gdbs:
    if 'SurveyPoint_HD' in fiona.listlayers(survey):
        survey_pts = gpd.read_file(survey, layer = 'SurveyPoint_HD')
    else:
        survey_pts = gpd.read_file(survey, layer = 'SurveyPoint')

    if len(survey_pts) < num_pts:
        print(f"Length of survey {survey.stem} less than 10k, select a different survey for processing")
        bad_surveys.append(survey)
        continue
    x, y, z, x_raw, y_raw, z_raw = processing_help.decimate_survey_points(survey_pts, tgt_num_pts=num_pts, reduction_method=reduction_method)
    
    TINinterpolator = TIN(
        tgt_survey = survey,
        reduction_method = reduction_method,
        x=x, y=y, z=z,
        grid_res=grid_res,
        tgt_num_pts=num_pts,
        plot_outputs=plots,
        n_jobs=4
    )

    tin_path = TINinterpolator.generate()
    _,_ = processing_help.validate_bathymetric_surface(tin_path, x_raw, y_raw, z_raw)

    NatNinterpolator = NaturalNeighbor(
        tgt_survey = survey, 
        reduction_method = reduction_method,
        x=x, y=y, z=z, 
        grid_res=grid_res,
        tgt_num_pts=num_pts,
        plot_outputs=plots,
        n_jobs=4
    )
    
    natn_path = NatNinterpolator.generate()
    _,_ = processing_help.validate_bathymetric_surface(natn_path, x_raw, y_raw, z_raw)

    IDWinterpolator = IDW(
        tgt_survey=survey,
        reduction_method=reduction_method,
        x=x, y=y, z=z,
        grid_res=grid_res,
        tgt_num_pts=num_pts,
        plot_outputs=plots,
        n_jobs=4
    )
    
    idw_path = IDWinterpolator.generate()
    _,_ = processing_help.validate_bathymetric_surface(idw_path, x_raw, y_raw, z_raw)

    RBFinterpolator = RBF(
        tgt_survey=survey,
        reduction_method=reduction_method,
        x=x, y=y, z=z, 
        grid_res=grid_res,
        tgt_num_pts=num_pts,
        plot_outputs=plots,
        n_jobs=4
    )

    rbf_path = RBFinterpolator.generate(auto_tune=True)
    _,_ = processing_help.validate_bathymetric_surface(rbf_path, x_raw, y_raw, z_raw)

    for detrend_bool in [False, True]:
        
        isoKrige_AIC = isoAutoKrige(
                        tgt_survey = survey, 
                        vario_models=vario_models,
                        reduction_method=reduction_method,
                        x=x, y=y, z=z, 
                        spline_damping=1e-1,                                            
                        vario_estimator = estimator,            
                        chunked_kriging = True,
                        grid_res=grid_res,
                        detrend=detrend_bool,
                        tgt_num_pts=num_pts,
                        plot_outputs=plots,
                        n_jobs=4
                )
        krige_path_AIC, _ = isoKrige_AIC.generate()
        _,_ = processing_help.validate_bathymetric_surface(krige_path_AIC, x_raw, y_raw, z_raw)


In [ ]:
# TODO: substitute these surveys for ones with larger than 10k points for our study
# need to go back and redo the categorize step
print(bad_surveys)